# Gold — Date Dimension
Generates a comprehensive daily date spine from 1980-01-01 to 2030-12-31.

**Output:** `gold.dim_date` (Delta table)  
**Grain:** One row per calendar day  
**Surrogate key:** `date_key` (YYYYMMDD)

In [ ]:
df_dim_date = spark.sql("""
    SELECT
        YEAR(date) * 10000 + MONTH(date) * 100 + DAY(date) AS date_key,
        date AS full_date,
        DAYOFWEEK(date) AS day_of_week_num,
        DATE_FORMAT(date, 'EEEE') AS day_name,
        DAY(date) AS day_of_month,
        DAYOFYEAR(date) AS day_of_year,
        WEEKOFYEAR(date) AS week_of_year,
        MONTH(date) AS month_num,
        DATE_FORMAT(date, 'MMMM') AS month_name,
        DATE_FORMAT(date, 'MMM') AS month_short_name,
        QUARTER(date) AS quarter_num,
        CONCAT(YEAR(date), ' Q', QUARTER(date)) AS quarter_label,
        YEAR(date) AS year_num,
        YEAR(date) * 100 + MONTH(date) AS year_month_key,
        CASE WHEN DAYOFWEEK(date) NOT IN (1,7) THEN TRUE ELSE FALSE END AS is_weekday,
        CASE WHEN DAYOFWEEK(date) IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend,
        CASE WHEN DAY(date) = 1 THEN TRUE ELSE FALSE END AS is_month_start,
        CASE WHEN date = LAST_DAY(date) THEN TRUE ELSE FALSE END AS is_month_end,
        CASE WHEN DAY(date) = 1 AND MONTH(date) IN (1, 4, 7, 10) THEN TRUE ELSE FALSE END AS is_quarter_start,
        CASE WHEN date = LAST_DAY(date) AND MONTH(date) IN (3, 6, 9, 12) THEN TRUE ELSE FALSE END AS is_quarter_end,
        CASE WHEN DAY(date) = 1 AND MONTH(date) = 1 THEN TRUE ELSE FALSE END AS is_year_start,
        CASE WHEN DAY(date) = 31 AND MONTH(date) = 12 THEN TRUE ELSE FALSE END AS is_year_end,
        CASE
            WHEN date BETWEEN '2008-01-01' AND '2011-12-31' THEN 'Banking Collapse'
            WHEN date BETWEEN '2020-01-01' AND '2021-12-31' THEN 'Pandemic'
            WHEN date BETWEEN '2022-01-01' AND '2026-04-30' THEN 'Inflation & Volcanic'
            ELSE 'Normal'
        END AS crisis_period,
        CASE
            WHEN date BETWEEN '2008-01-01' AND '2011-12-31' THEN TRUE
            WHEN date BETWEEN '2020-01-01' AND '2021-12-31' THEN TRUE
            WHEN date BETWEEN '2022-01-01' AND '2026-04-30' THEN TRUE
            ELSE FALSE
        END AS is_crisis_period,
        CURRENT_TIMESTAMP() AS refreshed_at
    FROM (
        SELECT EXPLODE(SEQUENCE(
            DATE('1980-01-01'),
            DATE('2030-12-31'),
            INTERVAL 1 DAY
        )) AS date
    )
""")

In [ ]:
df_dim_date.createOrReplaceTempView("v_dim_date")

if not spark.catalog.tableExists("gold.dim_date"):
    df_dim_date.write.format("delta").saveAsTable("gold.dim_date")
else:
    spark.sql("""
        MERGE INTO gold.dim_date AS target
        USING v_dim_date AS source
        ON target.date_key = source.date_key
        WHEN MATCHED THEN UPDATE SET
            target.full_date        = source.full_date,
            target.refreshed_at     = source.refreshed_at,
            target.is_quarter_start = source.is_quarter_start,
            target.is_quarter_end   = source.is_quarter_end,
            target.is_year_start    = source.is_year_start,
            target.is_year_end      = source.is_year_end,
            target.crisis_period    = source.crisis_period,
            target.is_crisis_period = source.is_crisis_period
        WHEN NOT MATCHED THEN INSERT (
            date_key, full_date, day_of_week_num, day_name, day_of_month, day_of_year,
            week_of_year, month_num, month_name, month_short_name, quarter_num, quarter_label,
            year_num, year_month_key, is_weekday, is_weekend, is_month_start, is_month_end,
            is_quarter_start, is_quarter_end, is_year_start, is_year_end,
            crisis_period, is_crisis_period, refreshed_at
        )
        VALUES (
            source.date_key, source.full_date, source.day_of_week_num, source.day_name,
            source.day_of_month, source.day_of_year, source.week_of_year, source.month_num,
            source.month_name, source.month_short_name, source.quarter_num, source.quarter_label,
            source.year_num, source.year_month_key, source.is_weekday, source.is_weekend,
            source.is_month_start, source.is_month_end,
            source.is_quarter_start, source.is_quarter_end, source.is_year_start, source.is_year_end,
            source.crisis_period, source.is_crisis_period, source.refreshed_at
        )
    """)

print(f"Dimension updated: {spark.table('gold.dim_date').count()} rows.")